# 02 - التصور البصري للتضمينات وتشابه جيب التمام
## (Visualization of Embeddings & Cosine Similarity)

يركز هذا الدفتر على تطبيق كافة المفاهيم المطروحة في المحاضرة 20 من الكورس:
1. **تمثيل الكلمات في فضاء ثنائي الأبعاد (2D Toy Embeddings)** وتصورها بصرياً عبر Matplotlib.
2. **التكتل الدلالي (Semantic Clustering)** لمجموعات الكلمات (القطط، الكلاب، المركبات).
3. **بناء دالة تشابه جيب التمام (Cosine Similarity)** بالاعتماد على الضرب النقطي والمعيار الإقليدي عبر NumPy.
4. **حساب مصفوفة التشابه الشاملة (Pairwise Similarity Matrix)** لكافة الكلمات.
5. **إسقاط متجهات حقيقية متعددة الأبعاد (384D) باستخدام تقليل الأبعاد (PCA)** لإثبات أن النماذج الحقيقية تتبع نفس المبدأ الهندسي.

---
### 1. تمثيل الكلمات ثنائية الأبعاد ورسمها بيانياً (2D Embeddings & Visualization)
نقوم بتعريف الكلمات الست المذكورة في المحاضرة كإحداثيات [x, y] ورسمها لنرى التكتل الدلالي.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. تعريف إحداثيات الكلمات المبسطة كما وردت في المحاضرة
embeddings_2d = {
    "cat": np.array([0.80, 0.60]),
    "kitten": np.array([0.75, 0.65]),
    "dog": np.array([0.70, 0.30]),
    "puppy": np.array([0.65, 0.35]),
    "car": np.array([-0.50, 0.80]),
    "truck": np.array([-0.60, 0.70])
}

# 2. رسم الكلمات في فضاء ثنائي الأبعاد
plt.figure(figsize=(10, 7))

colors = {
    "cat": "#e74c3c", "kitten": "#e74c3c",
    "dog": "#2980b9", "puppy": "#2980b9",
    "car": "#27ae60", "truck": "#27ae60"
}

for word, coord in embeddings_2d.items():
    x, y = coord[0], coord[1]
    plt.scatter(x, y, color=colors[word], s=200, zorder=5)
    plt.annotate(
        f"{word} ({x:.2f}, {y:.2f})",
        xy=(x, y),
        xytext=(x + 0.03, y + 0.02),
        fontsize=11,
        fontweight="bold",
        color=colors[word]
    )
    # رسم سهم متجه من نقطة الأصل (0,0)
    plt.arrow(0, 0, x, y, color=colors[word], alpha=0.25, width=0.005, head_width=0.03, length_includes_head=True)

plt.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.axvline(0, color="gray", linestyle="--", alpha=0.5)
plt.title("Simplified 2D Word Embeddings Space", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Feature Dimension 1 (X-axis)", fontsize=12)
plt.ylabel("Feature Dimension 2 (Y-axis)", fontsize=12)
plt.xlim(-1.0, 1.2)
plt.ylim(-0.2, 1.1)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

---
### 2. بناء دالة حساب تشابه جيب التمام (Cosine Similarity Implementation)
$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$
* **+1:** تطابق دلالي كامل في نفس الاتجاه.
* **0:** تعامد تام (لا توجد أي صلة دلالية).
* **-1:** اتجاهان متعاكسان تماماً في الفضاء.

In [ ]:
def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """حساب تشابه جيب التمام بين متجهين باستخدام NumPy."""
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
        
    return float(dot_product / (norm_a * norm_b))

# حساب الأمثلة المذكورة في المحاضرة
sim_cat_kitten = cosine_similarity(embeddings_2d["cat"], embeddings_2d["kitten"])
sim_cat_car = cosine_similarity(embeddings_2d["cat"], embeddings_2d["car"])
sim_dog_puppy = cosine_similarity(embeddings_2d["dog"], embeddings_2d["puppy"])
sim_car_truck = cosine_similarity(embeddings_2d["car"], embeddings_2d["truck"])

print(f"🐾 Cosine Similarity (cat <-> kitten): {sim_cat_kitten:.4f}  (Very High Similarity)")
print(f"🐶 Cosine Similarity (dog <-> puppy):  {sim_dog_puppy:.4f}  (Very High Similarity)")
print(f"🚗 Cosine Similarity (car <-> truck):  {sim_car_truck:.4f}  (Very High Similarity)")
print(f"❌ Cosine Similarity (cat <-> car):    {sim_cat_car:.4f} (Negative / Unrelated)")

---
### 3. مصفوفة التشابه الكاملة (Pairwise Similarity Matrix)
حساب التشابه بين كل كلمة وباقي الكلمات دفعة واحدة لمقارنة العناقيد الدلالية.

In [ ]:
words = list(embeddings_2d.keys())
mat = np.array(list(embeddings_2d.values()))

# حساب المصفوفة المتجهية: Dot Products / (Norms * Norms.T)
dot_matrix = np.dot(mat, mat.T)
norms = np.linalg.norm(mat, axis=1, keepdims=True)
similarity_matrix = dot_matrix / (norms * norms.T)

print("📊 مصفوفة تشابه جيب التمام الكاملة:")
header = "          " + " ".join([f"{w:>9}" for w in words])
print(header)
print("-" * len(header))
for i, row in enumerate(similarity_matrix):
    row_str = f"{words[i]:<10}" + " ".join([f"{val:>9.4f}" for val in row])
    print(row_str)

---
### 4. تطبيق عملي متقدم: إسقاط متجهات حقيقية (384 أبعاد) عبر تقليل الأبعاد (PCA)
كما ذكر المحاضر، النماذج الحقيقية لا تنتج بعدين بل مئات أو آلاف الأبعاد.
هنا نقوم بتوليد تضمينات لـ 6 جمل باستخدام نموذج `all-MiniLM-L6-v2` الحقيقي (384 بُعداً)، ثم نستخدم **تحليل المكونات الرئيسية (PCA)** بواسطة NumPy لإسقاطها على بُعدين ورسمها.

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. تحميل النموذج وتوليد المتجهات عالية الأبعاد
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "A cute little kitten playing with yarn",
    "A domestic cat resting on the soft sofa",
    "A loyal dog barking at the mail carrier",
    "A playful puppy running in the park",
    "A sports car speeding on the highway",
    "A heavy duty cargo truck transporting freight"
]
labels = ["kitten", "cat", "dog", "puppy", "sports_car", "cargo_truck"]

real_embeddings = model.encode(sentences)
print(f"✅ تم استخراج متجهات حقيقية بأبعاد: {real_embeddings.shape} (6 جمل × 384 بعداً)\n")

# 2. تنفيذ PCA من الصفر عبر Singular Value Decomposition (SVD) في NumPy
def pca_numpy(X: np.ndarray, n_components: int = 2) -> np.ndarray:
    # توسيط البيانات بطرح المتوسط
    X_centered = X - np.mean(X, axis=0)
    # حساب تفكيك القيم الشاذة SVD
    U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    # الإسقاط على أول مركبتين رئيسيتين
    return np.dot(X_centered, Vt[:n_components].T)

projected_2d = pca_numpy(real_embeddings, n_components=2)

# 3. رسم المتجهات الحقيقية بعد إسقاطها إلى 2D
plt.figure(figsize=(10, 6))
category_colors = ["#e74c3c", "#e74c3c", "#2980b9", "#2980b9", "#27ae60", "#27ae60"]

for i, label in enumerate(labels):
    px, py = projected_2d[i, 0], projected_2d[i, 1]
    plt.scatter(px, py, color=category_colors[i], s=220, zorder=5)
    plt.annotate(
        f"{label}\n({sentences[i][:25]}...)",
        xy=(px, py),
        xytext=(px + 0.02, py + 0.01),
        fontsize=10,
        fontweight="bold",
        color=category_colors[i]
    )

plt.title("Real-World 384D Embeddings Projected to 2D via PCA", fontsize=13, fontweight="bold", pad=15)
plt.xlabel("Principal Component 1 (PC1)", fontsize=11)
plt.ylabel("Principal Component 2 (PC2)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 💡 الخلاصة والاستنتاج (Key Takeaways):
1. حتى في النماذج الواقعية متعددة الأبعاد (`384D`)، نجح نموذج التضمين في تجميع النصوص المتشابهة في نفس الحيز المكاني عند إسقاطها عبر PCA.
2. دالة تشابه جيب التمام (`Cosine Similarity`) هي القلب النابض لأنظمة البحث الدلالي في RAG، لأنها تحدد قرب المعنى بمعزل عن طول الفقرة أو عدد كلماتها.